# Assignment 5 · Task 2 — Part A: Human Matting

| | |
|---|---|
| **Architecture** | U-Net trained from scratch (no pretrained weights) |
| **Loss** | `0.5 × L1 + 0.5 × Dice` |
| **Target** | Test IoU ≥ 0.85 |
| **Dataset** | [AISegment Matting Human Datasets](https://www.kaggle.com/datasets/laurentmih/aisegmentcom-matting-human-datasets) |

**Steps:** GPU check → clone repo → install deps → verify dataset → configure → smoke test → train → evaluate → IoU report card → training curves → 5-sample visualisation (Input | GT α | Predicted α | Cutout) → zip outputs.

> **Before running:** set `GITHUB_REPO` in Cell 2 to your repo URL.

## 1 · Environment Check

In [ ]:
import subprocess, sys, os, shutil, re
from pathlib import Path

print('=' * 60)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
!df -h /kaggle/working

## 2 · Clone GitHub Repo
> **Edit `GITHUB_REPO`** to your actual repository URL before running.

In [ ]:
# ✏️  CHANGE THIS
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/CV-AS-5-TASK2'

REPO_DIR = Path('/kaggle/working/repo')
if REPO_DIR.exists():
    print('Repo present — pulling latest ...')
    !git -C {REPO_DIR} pull
else:
    !git clone --depth 1 {GITHUB_REPO} {REPO_DIR}

print('\nRepo contents:')
for p in sorted(REPO_DIR.iterdir()):
    print(f'  {p.name}')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('\n[OK] Repo ready.')

## 3 · Install Dependencies

In [ ]:
req = REPO_DIR / 'requirements.txt'
if req.exists():
    !pip install -q -r {req}
else:
    !pip install -q tqdm pyyaml Pillow
import yaml, tqdm
print(f'PyYAML {yaml.__version__} | tqdm {tqdm.__version__}')
print('[OK] Dependencies ready.')

## 4 · Verify Dataset Mount

In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display as ipy_display, Image as IPyImage

DATASET_ROOT = Path('/kaggle/input/aisegmentcom-matting-human-datasets')
assert DATASET_ROOT.exists(), (
    f'Dataset not found at {DATASET_ROOT}.\n'
    'Add it via: Notebook -> Data -> Add Dataset -> aisegmentcom-matting-human-datasets'
)

clip_imgs = list(DATASET_ROOT.glob('clip_img/**/*.jpg'))
mattes    = list(DATASET_ROOT.glob('matting/**/*.png'))
print(f'Clip images : {len(clip_imgs):,}')
print(f'Mattes      : {len(mattes):,}')
assert len(clip_imgs) > 0 and len(mattes) > 0

# Quick sanity-check visualisation
def _resolve_matte(p):
    parts = list(p.parts)
    ci = parts.index('clip_img')
    parts[ci] = 'matting'
    for i, pt in enumerate(parts):
        if pt.startswith('clip_'):
            parts[i] = pt.replace('clip_', 'matting_', 1); break
    parts[-1] = p.stem + '.png'
    return Path(*parts)

img0   = Image.open(clip_imgs[0]).convert('RGB')
mp0    = _resolve_matte(clip_imgs[0])
matte0 = (Image.open(mp0).split()[3] if mp0.exists() and Image.open(mp0).mode == 'RGBA'
          else Image.open(mp0).convert('L') if mp0.exists()
          else Image.new('L', img0.size, 128))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(np.array(img0)); axes[0].set_title('Sample RGB'); axes[0].axis('off')
axes[1].imshow(np.array(matte0), cmap='gray'); axes[1].set_title('Sample Matte'); axes[1].axis('off')
plt.suptitle('Dataset sanity check', fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/dataset_sample.png', dpi=120, bbox_inches='tight')
plt.close()
ipy_display(IPyImage('/kaggle/working/dataset_sample.png'))
print('[OK] Dataset verified.')

## 5 · Configure for Kaggle (T4 overrides)

In [ ]:
import yaml

with open(REPO_DIR / 'config.yaml') as fh:
    cfg = yaml.safe_load(fh)

cfg['data']['dataset_root']       = str(DATASET_ROOT)
cfg['data']['output_dir']         = '/kaggle/working/outputs'
cfg['matting']['weights_path']    = '/kaggle/working/matting_weights.pth'
cfg['matting']['num_workers']     = 4
cfg['matting']['batch_size']      = 16
cfg['matting']['epochs']          = 25
cfg['matting']['train_size']      = 5000
cfg['matting']['val_size']        = 500
cfg['matting']['test_size']       = 500

KAGGLE_CFG = Path('/kaggle/working/kaggle_config.yaml')
with open(KAGGLE_CFG, 'w') as fh:
    yaml.dump(cfg, fh, default_flow_style=False, sort_keys=False)

Path('/kaggle/working/outputs').mkdir(exist_ok=True)
print('Config written to', KAGGLE_CFG)
print(f"  arch        : {cfg['matting']['architecture']}")
print(f"  input_size  : {cfg['matting']['input_size']}")
print(f"  epochs      : {cfg['matting']['epochs']}")
print(f"  batch_size  : {cfg['matting']['batch_size']}")
print(f"  loss        : {cfg['matting']['l1_weight']}*L1 + {cfg['matting']['dice_weight']}*Dice")
print('[OK] Config ready.')

## 6 · Smoke-Test Both Architectures

In [ ]:
import os; os.chdir(REPO_DIR)
!python model.py

## 7 · Train the Matting Model

Expected time on **T4 GPU**: ~2–4 min/epoch → ~50–100 min for 25 epochs.  
Early stopping (patience = 5) will stop sooner if val IoU plateaus.

In [ ]:
train_cmd = [sys.executable, str(REPO_DIR / 'train.py'), '--config', str(KAGGLE_CFG)]
print('Command:', ' '.join(train_cmd))
print('=' * 60)

proc = subprocess.Popen(
    train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, cwd=str(REPO_DIR))
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'train.py exited with code {proc.returncode}')
print('\n[OK] Training complete.')

## 8 · Evaluate on Test Split

In [ ]:
eval_cmd = [
    sys.executable, str(REPO_DIR / 'evaluate.py'),
    '--weights', '/kaggle/working/matting_weights.pth',
    '--config',  str(KAGGLE_CFG),
    '--split',   'test',
]
print('Command:', ' '.join(eval_cmd))
print('=' * 60)

eval_lines = []
proc = subprocess.Popen(
    eval_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, cwd=str(REPO_DIR))
for line in proc.stdout:
    print(line, end='', flush=True)
    eval_lines.append(line)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'evaluate.py exited with code {proc.returncode}')

# Parse test IoU for downstream cells
TEST_IOU = None
for line in eval_lines:
    m = re.search(r'IoU\s*[:|]\s*([0-9]+\.[0-9]+)', line)
    if m:
        TEST_IOU = float(m.group(1))
        break
print(f'\nParsed TEST_IOU = {TEST_IOU}')
print('[OK] Evaluation complete.')

## 9 · IoU Report Card

Displays the test-split IoU prominently against the ≥ 0.85 target.

In [ ]:
import pandas as pd
from IPython.display import display as ipy_display, HTML

log_csv      = Path('/kaggle/working/outputs/matting_train_log.csv')
df_log       = pd.read_csv(log_csv)
best_val_iou = df_log['val_iou'].max()
best_epoch   = int(df_log.loc[df_log['val_iou'].idxmax(), 'epoch'])

target_met  = (TEST_IOU is not None and TEST_IOU >= 0.85)
badge_color = '#276749' if target_met else '#9B2335'
status      = 'ACHIEVED' if target_met else 'NOT YET'
iou_str     = f'{TEST_IOU:.4f}' if TEST_IOU is not None else 'N/A'
arch        = cfg['matting']['architecture'].upper()

html = f"""
<div style='font-family:monospace;background:#1A1A2E;color:white;
            padding:20px;border-radius:10px;margin:10px 0'>
  <h2 style='color:#68D391;margin-bottom:14px'>Matting Model &mdash; Test Results (Part A)</h2>
  <table style='border-collapse:collapse;width:100%;font-size:15px'>
    <tr><td style='padding:8px 16px;color:#A0AEC0'>Architecture</td>
        <td style='padding:8px 16px;color:white;font-weight:bold'>{arch}</td></tr>
    <tr style='background:rgba(255,255,255,.05)'>
        <td style='padding:8px 16px;color:#A0AEC0'>Best Val IoU</td>
        <td style='padding:8px 16px;color:#68D391;font-weight:bold'>{best_val_iou:.4f}&nbsp;(epoch {best_epoch})</td></tr>
    <tr><td style='padding:8px 16px;color:#A0AEC0'>Test IoU</td>
        <td style='padding:8px 16px;color:#FBD38D;font-weight:bold;font-size:20px'>{iou_str}</td></tr>
    <tr style='background:rgba(255,255,255,.05)'>
        <td style='padding:8px 16px;color:#A0AEC0'>Target</td>
        <td style='padding:8px 16px;color:white'>&ge; 0.85</td></tr>
    <tr><td style='padding:8px 16px;color:#A0AEC0'>Loss</td>
        <td style='padding:8px 16px;color:white'>0.5 &times; L1 + 0.5 &times; Dice</td></tr>
  </table>
  <div style='margin-top:16px;padding:12px 20px;background:{badge_color};
              border-radius:6px;font-size:17px;font-weight:bold;text-align:center'>
    Test IoU = {iou_str} &mdash; Target {status}
  </div>
</div>
"""
ipy_display(HTML(html))

## 10 · Training Curves

In [ ]:
log_csv = Path('/kaggle/working/outputs/matting_train_log.csv')
assert log_csv.exists(), f'Log not found: {log_csv}'

curves_cmd = [
    sys.executable, str(REPO_DIR / 'visualise.py'),
    'curves',
    '--log', str(log_csv),
    '--out', '/kaggle/working/outputs/training_curves.png',
]
proc = subprocess.Popen(curves_cmd, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, cwd=str(REPO_DIR))
for line in proc.stdout: print(line, end='')
proc.wait()

from IPython.display import Image as IPyImage
ipy_display(IPyImage('/kaggle/working/outputs/training_curves.png'))

## 11 · 5-Sample Frame Visualisation

Each row shows one sample from the dataset across **4 columns**:

| Column | Content |
|---|---|
| Input RGB | Original portrait image |
| Ground-truth α | Dataset alpha matte |
| Predicted α | Model output matte |
| Cutout | Subject composited over white background |

The test-split IoU is embedded as a coloured badge at the bottom of the figure.

In [ ]:
preds_cmd = [
    sys.executable, str(REPO_DIR / 'visualise.py'),
    'predictions',
    '--weights', '/kaggle/working/matting_weights.pth',
    '--config',  str(KAGGLE_CFG),
    '--n',       '5',
    '--out',     '/kaggle/working/outputs/sample_predictions.png',
]
# Embed the test IoU as a badge on the figure
if TEST_IOU is not None:
    preds_cmd += ['--iou', str(round(TEST_IOU, 4))]

print('Command:', ' '.join(preds_cmd))
proc = subprocess.Popen(preds_cmd, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, cwd=str(REPO_DIR))
for line in proc.stdout: print(line, end='')
proc.wait()

from IPython.display import Image as IPyImage
ipy_display(IPyImage('/kaggle/working/outputs/sample_predictions.png'))

## 12 · Epoch Log Table

In [ ]:
import pandas as pd
from IPython.display import display as ipy_display

df    = pd.read_csv('/kaggle/working/outputs/matting_train_log.csv')
best  = df['val_iou'].idxmax()

def hl(s):
    return ['background-color:#d4edda;font-weight:bold' if i == best else '' for i in s.index]

print(f"Best val IoU = {df['val_iou'].max():.4f}  (epoch {int(df.loc[best,'epoch'])})")
ipy_display(
    df.style.apply(hl, axis=0)
      .format({'train_loss':'{:.4f}','val_loss':'{:.4f}','val_iou':'{:.4f}','lr':'{:.2e}'})
      .set_caption('Training log — green row = best val IoU')
)

## 13 · Package Outputs for Download

In [ ]:
import shutil, os
outputs = Path('/kaggle/working/outputs')
weights = Path('/kaggle/working/matting_weights.pth')
if weights.exists():
    shutil.copy(weights, outputs / 'matting_weights.pth')

zip_base = '/kaggle/working/part_a_results'
shutil.make_archive(zip_base, 'zip', outputs)
size_mb = os.path.getsize(zip_base + '.zip') / 1e6

print(f'Archive: {zip_base}.zip  ({size_mb:.1f} MB)')
print('Contents:')
for p in sorted(outputs.iterdir()):
    print(f'  {p.name:<42} {os.path.getsize(p)/1e6:.2f} MB')
print('\nDownload: Kaggle sidebar -> Output -> part_a_results.zip')

---
## Done

| Output file | Description |
|---|---|
| `matting_weights.pth` | Best model checkpoint |
| `matting_train_log.csv` | Per-epoch loss & IoU |
| `training_curves.png` | Loss + IoU curves (target line at 0.85) |
| `sample_predictions.png` | 5-sample grid: Input / GT α / Predicted α / Cutout |
| `dataset_sample.png` | Sanity-check dataset pair |

All bundled in **`part_a_results.zip`** in the Output panel.